# Unified ExperimentBu notebook `experiments/run_unified.py` uzerinden tek deney kosusu calistirir.Colab icin genelde `free_colab_t4` ile basla.

In [ ]:
import subprocessimport sysfrom pathlib import Pathcandidate_roots = [    Path.cwd(),    Path.cwd().parent,    Path('/content/LLMComparison'),    Path('/content/drive/MyDrive/LLMComparison'),]PROJECT_ROOT = next(    (        root        for root in candidate_roots        if (root / 'experiments').exists() and (root / 'src').exists()    ),    None,)if PROJECT_ROOT is None:    raise FileNotFoundError('Project root not found. Clone into /content/LLMComparison or mount Drive first.')if str(PROJECT_ROOT) not in sys.path:    sys.path.insert(0, str(PROJECT_ROOT))print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from datetime import datetimePRESET = 'free_colab_t4'MODELS = ['generalist']DATASETS = ['hf_vqa_rad']NUM_SAMPLES = 50OUTPUT_DIR = PROJECT_ROOT / 'results'RUN_NAME = f"unified_{datetime.now().strftime('%Y%m%d_%H%M')}"print('Preset:', PRESET)print('Run name:', RUN_NAME)

In [ ]:
import reimport timecommand = [    sys.executable,    str(PROJECT_ROOT / 'experiments' / 'run_unified.py'),    '--preset', PRESET,    '--models', *MODELS,    '--datasets', *DATASETS,    '--num-samples', str(NUM_SAMPLES),    '--skip-inaccessible',    '--output-dir', str(OUTPUT_DIR),    '--run-name', RUN_NAME,]print('Running command:')print(command)total_steps = max(1, len(MODELS) * len(DATASETS))completed_steps = 0step_markers_seen = set()start_ts = time.time()process = subprocess.Popen(    command,    cwd=PROJECT_ROOT,    stdout=subprocess.PIPE,    stderr=subprocess.STDOUT,    text=True,    bufsize=1,)for line in process.stdout:    text_line = line.rstrip()    marker_match = re.search(r'Running model=(.*?) dataset=(.*)', text_line)    if marker_match:        marker = marker_match.group(0)        if marker not in step_markers_seen:            step_markers_seen.add(marker)            completed_steps += 1    elapsed = time.time() - start_ts    elapsed_min = elapsed / 60.0    if completed_steps > 0:        avg_per_step = elapsed / completed_steps        remaining_steps = max(0, total_steps - completed_steps)        eta_sec = avg_per_step * remaining_steps        eta_min = eta_sec / 60.0        print(f'[{completed_steps}/{total_steps}] elapsed={elapsed_min:.1f}m eta~{eta_min:.1f}m | {text_line}')    else:        print(f'[0/{total_steps}] elapsed={elapsed_min:.1f}m eta~unknown | {text_line}')return_code = process.wait()total_elapsed_min = (time.time() - start_ts) / 60.0print(f'Finished with code={return_code} in {total_elapsed_min:.1f}m')if return_code != 0:    raise RuntimeError(f'run_unified failed with exit code {return_code}')

In [ ]:
import jsonimport pandas as pdresults_dir = OUTPUT_DIR / RUN_NAMEaggregate_path = results_dir / 'aggregate_metrics.json'if aggregate_path.exists():    payload = json.loads(aggregate_path.read_text())    rows = [        {'model_name': model_name, **metrics}        for model_name, metrics in payload.get('metrics', {}).items()    ]    display(pd.DataFrame(rows))else:    print('aggregate_metrics.json not found yet.')